# Multi-Seed Validation — Gemma-2-9B | CB2 | Stage 4
## Seeds: 123 and 456 (seed=42 already complete: F1=0.8959)
**Purpose:** Validate CB2 champion result is robust across seeds.

**Protocol:**
- Data split: **always seed=42** (same test set as all stages)
- Training seed: **123, 456** (changes weight init, dropout, data shuffling)
- Model: `google/gemma-2-9b-it` | Rank: **r=16** (CB2 champion) | Formula: Wf = W(I+UV)+AB
- Metrics: Macro F1 + MCC + AUC-ROC

**Seed 42 result (already completed):** F1=0.8959 | MCC=0.7949 | AUC=0.9695  
**Targets:** S2=0.8885 | S3=0.8948


---
## Step 1: Install

**Restart runtime after running, then Step 2 onwards.**

In [1]:
!pip install -q \
    "transformers>=4.47.0,<4.52.0" \
    "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" \
    "accelerate>=1.2.0" \
    "datasets>=3.0.0" \
    "numpy<2.1" \
    "pandas<3.0" \
    "torchvision==0.20.1" \
    "scikit-learn" \
    tqdm matplotlib seaborn
print("✓ Installed. Restart runtime, then run from Step 2.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 144.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## Step 2: Imports

In [2]:
import os, random, collections, re
from typing import Dict, List
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt, seaborn as sns

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
    balanced_accuracy_score, matthews_corrcoef, roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments, Trainer, TrainerCallback,
)
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
import bitsandbytes.functional as bnb_F
import bitsandbytes as bnb
from google.colab import drive, userdata

def set_all_seeds(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_all_seeds(42)
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print("✓ HF token loaded.")
drive.mount('/content/drive')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {device} | {torch.cuda.get_device_name(0)}")


✓ HF token loaded.
Mounted at /content/drive
✓ Device: cuda | NVIDIA A100-SXM4-80GB


---
## Step 3: Configuration — CB2 Multi-Seed

In [3]:
MODEL_NAME   = "google/gemma-2-9b-it"
NUM_LABELS   = 2          # Binary
MAX_LENGTH   = 300        # CB2 conversations
LORA_RANK    = 16         # CB2 champion rank
LORA_ALPHA   = 16
LORA_DROPOUT = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj']

NUM_EPOCHS       = 1
LEARNING_RATE    = 2e-4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE  = 32
WEIGHT_DECAY     = 0.01
EVAL_STEPS       = 100    # CB2: more frequent
LOGGING_STEPS    = 50
OUTPUT_DIR       = "/content/drive/MyDrive/Results/CB2_GeneralisedLoRA_Gemma2_9B_MultiSeed"

CB2_PATH             = "/content/drive/MyDrive/Datasets/cb2.csv"
SPLIT_SEED           = 42
SIMILARITY_THRESHOLD = 0.90
TRAIN_SEEDS          = [123, 456]

LABEL_MAPPING = {0: 'not_cyberbullying', 1: 'cyberbullying'}
ID2LABEL      = LABEL_MAPPING
TARGET_NAMES  = ['not_cyberbullying', 'cyberbullying']

print("✓ CB2 Multi-seed config loaded.")
print(f"  Model      : {MODEL_NAME}")
print(f"  Task       : Binary (CB2) | r={LORA_RANK}")
print(f"  Split seed : {SPLIT_SEED} (fixed)")
print(f"  Train seeds: {TRAIN_SEEDS}")
print(f"  Seed 42    : F1=0.8959 | S2=0.8885 | S3=0.8948")


✓ CB2 Multi-seed config loaded.
  Model      : google/gemma-2-9b-it
  Task       : Binary (CB2) | r=16
  Split seed : 42 (fixed)
  Train seeds: [123, 456]
  Seed 42    : F1=0.8959 | S2=0.8885 | S3=0.8948


---
## Steps 4–6: Load, Clean, Split CB2

In [4]:
def preprocess_text_for_llm(text, remove_hashtags=False, remove_emojis=False):
    if not isinstance(text, str): return ""
    text = re.sub(r'\bRT\s+@\w+:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'http\S+|www\.\S+', '[URL]', text)
    text = re.sub(r'@\w+', '[USER]', text)
    if remove_hashtags: text = re.sub(r'#(\w+)', r'\1', text)
    if remove_emojis: text = text.encode('ascii', 'ignore').decode('ascii')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'([!?.])\1+', r'\1', text)
    if len(text.split()) < 3: return ""
    return text

df = pd.read_csv(CB2_PATH)
TEXT_COL = 'Message'; LABEL_COL = 'Label'
df = df.rename(columns={TEXT_COL: 'text', LABEL_COL: 'label'})
df = df[['text', 'label']].copy()
corrupt_mask = df['text'].astype(str).str.strip().isin(['#NAME?','#REF!','#VALUE!','#N/A','nan',''])
df = df[~corrupt_mask].reset_index(drop=True)
df['label'] = df['label'].astype(int)
df['text'] = df['text'].apply(preprocess_text_for_llm)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)
df = df.drop_duplicates(subset=['text'], keep='first').reset_index(drop=True)
df = df[df['text'].str.split().str.len() >= 3].reset_index(drop=True)
print(f"Dataset: {len(df):,} samples")

# Split with FIXED seed=42
train_val_df, test_df = train_test_split(
    df, test_size=0.25, random_state=SPLIT_SEED, stratify=df['label'])

all_texts = list(train_val_df['text'].values) + list(test_df['text'].values)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=2, max_df=0.95)
tfidf_matrix = vectorizer.fit_transform(all_texts)
train_vecs = tfidf_matrix[:len(train_val_df)]
test_vecs = tfidf_matrix[len(train_val_df):]
leaked_indices = []
for start in range(0, len(test_df), 500):
    end = min(start+500, len(test_df))
    batch_sims = cosine_similarity(test_vecs[start:end], train_vecs)
    leaked_mask = batch_sims.max(axis=1) >= SIMILARITY_THRESHOLD
    leaked_indices.extend(test_df.index[start:end][leaked_mask].tolist())
if leaked_indices:
    leaked = test_df.loc[leaked_indices]
    test_df = test_df.drop(leaked_indices).reset_index(drop=True)
    train_val_df = pd.concat([train_val_df, leaked], ignore_index=True)
else:
    train_val_df = train_val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.10, random_state=SPLIT_SEED, stratify=train_val_df['label'])
train_df = train_df.reset_index(drop=True); val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"✓ Split seed={SPLIT_SEED} — identical test set to all stages.")


Dataset: 89,525 samples
Train: 62,261 | Val: 6,918 | Test: 20,346
✓ Split seed=42 — identical test set to all stages.


---
## Steps 7–10: Helpers, Metrics, Trainer

In [5]:
def get_class_weights(df, label_col='label'):
    counts = df[label_col].value_counts().sort_index()
    total = len(df); n_classes = len(counts)
    weights = [total / (n_classes * counts.get(i, 1)) for i in range(n_classes)]
    return [w / sum(weights) * n_classes for w in weights]

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average='macro', zero_division=0),
        "mcc": matthews_corrcoef(labels, predictions),
    }

class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float32) if class_weights is not None else None
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels"); outputs = model(**inputs); logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device)) if self.class_weights is not None else nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

print("✓ Helpers defined.")


✓ Helpers defined.


---
## Step 11: GeneralisedLoRALayer + Training Function

In [6]:
class GeneralisedLoRALayer(nn.Module):
    def __init__(self, base_layer, rank, dropout=0.05, alpha=None):
        super().__init__()
        self.rank = rank
        self.alpha = alpha if alpha is not None else rank
        self.scaling = self.alpha / self.rank
        inner = base_layer.base_layer if hasattr(base_layer, 'base_layer') else base_layer
        if hasattr(inner, 'in_features'):
            n, m = inner.in_features, inner.out_features
        elif hasattr(inner, 'weight'):
            n, m = inner.weight.shape[1], inner.weight.shape[0]
        else:
            raise ValueError(f"Cannot determine dims from {type(inner)}")
        self.n = n; self.m = m; self.inner = inner
        _dev = self.inner.weight.device
        self.lora_U = nn.Parameter(torch.empty(n, rank, device=_dev))
        self.lora_V = nn.Parameter(torch.zeros(rank, n, device=_dev))
        self.lora_A = nn.Parameter(torch.empty(m, rank, device=_dev))
        self.lora_B = nn.Parameter(torch.zeros(rank, n, device=_dev))
        nn.init.normal_(self.lora_U, mean=0.0, std=0.01)
        nn.init.normal_(self.lora_A, mean=0.0, std=0.01)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()
        for param in self.inner.parameters(): param.requires_grad = False

    def forward(self, x):
        base_out = self.inner(x)
        x_drop = self.dropout(x)
        lora_vx = x_drop @ self.lora_V.T
        lora_uvx = lora_vx @ self.lora_U.T
        mult_out = self.inner(lora_uvx)
        lora_bx = x_drop @ self.lora_B.T
        add_out = lora_bx @ self.lora_A.T
        return base_out + self.scaling * mult_out + self.scaling * add_out


def apply_generalised_lora_over_peft(model, target_modules, rank, dropout=0.05, alpha=None):
    from peft.tuners.lora import LoraLayer
    if alpha is None: alpha = rank
    replaced = 0
    module_dict = dict(model.named_modules())
    for name, module in list(model.named_modules()):
        if not isinstance(module, LoraLayer): continue
        if not any(name.endswith(f'.{t}') or name == t for t in target_modules): continue
        parts = name.rsplit('.', 1)
        if len(parts) == 2:
            parent = module_dict.get(parts[0])
            if parent is None: continue
            attr_name = parts[1]
        else:
            parent = model; attr_name = name
        setattr(parent, attr_name, GeneralisedLoRALayer(module, rank=rank, dropout=dropout, alpha=alpha))
        replaced += 1
    print(f"  ✓ Replaced {replaced} layers")
    return model


def run_single_seed_cb2(train_df, val_df, test_df, train_seed):
    set_all_seeds(train_seed)
    label = f'seed_{train_seed}'
    print(f'\n{"="*65}')
    print(f'  MULTI-SEED CB2 — train_seed={train_seed}')
    print(f'  Split seed=42 (fixed) | Gemma-2-9B | r={LORA_RANK}')
    print(f'{"="*65}')

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_dir = os.path.join(OUTPUT_DIR, label)
    os.makedirs(run_dir, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    def tokenise(examples):
        return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=MAX_LENGTH)

    train_m = train_df.copy(); val_m = val_df.copy(); test_m = test_df.copy()
    hf_train = Dataset.from_pandas(train_m[['text','label']].reset_index(drop=True)).map(tokenise, batched=True, remove_columns=['text'])
    hf_val = Dataset.from_pandas(val_m[['text','label']].reset_index(drop=True)).map(tokenise, batched=True, remove_columns=['text'])
    hf_train.set_format('torch'); hf_val.set_format('torch')

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
        lora_dropout=LORA_DROPOUT, bias='none', task_type='SEQ_CLS')

    print('[1] Loading model...')
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, quantization_config=quant_config, num_labels=NUM_LABELS,
        torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN)
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, lora_config)

    print('[2] Injecting GeneralisedLoRALayer...')
    model = apply_generalised_lora_over_peft(
        model, TARGET_MODULES, rank=LORA_RANK, dropout=LORA_DROPOUT, alpha=LORA_ALPHA)

    class_weights = get_class_weights(train_df)

    training_args = TrainingArguments(
        output_dir=run_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        eval_strategy='steps', eval_steps=EVAL_STEPS,
        logging_steps=LOGGING_STEPS, save_strategy='no',
        fp16=False, bf16=True, report_to='none', remove_unused_columns=False)

    trainer = CustomTrainer(
        model=model, args=training_args,
        train_dataset=hf_train, eval_dataset=hf_val,
        compute_metrics=compute_metrics, class_weights=class_weights)

    print(f'[3] Training (seed={train_seed})...')
    train_result = trainer.train()
    runtime = train_result.metrics.get('train_runtime', 0)
    print(f'    Time: {runtime:.0f}s ({runtime/3600:.2f} hrs)')

    print('[4] Evaluating...')
    model.eval()
    dev = next(model.parameters()).device
    all_preds, all_probs = [], []
    texts = test_m['text'].tolist()
    for start in range(0, len(texts), 32):
        enc = tokenizer(texts[start:start+32], padding='max_length',
                        truncation=True, max_length=MAX_LENGTH, return_tensors='pt').to(dev)
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits.float(), dim=-1)
        all_preds.extend(torch.argmax(logits, dim=-1).cpu().numpy().tolist())
        all_probs.extend(probs[:, 1].cpu().numpy().tolist())

    y_true = test_m['label'].tolist()
    macro_f1 = f1_score(y_true, all_preds, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y_true, all_preds)
    accuracy = accuracy_score(y_true, all_preds)
    try:
        auc = roc_auc_score(y_true, all_probs)
    except:
        auc = 0.0

    result = {'seed': train_seed, 'macro_f1': macro_f1, 'mcc': mcc,
              'auc_roc': auc, 'accuracy': accuracy, 'train_runtime_s': runtime}

    pd.DataFrame([result]).to_csv(os.path.join(run_dir, f'result_seed{train_seed}.csv'), index=False)
    print(f'\n    Seed {train_seed}: F1={macro_f1:.4f} | MCC={mcc:.4f} | AUC={auc:.4f}')

    del model, trainer; torch.cuda.empty_cache()
    return result

print("✓ run_single_seed_cb2() defined.")


✓ run_single_seed_cb2() defined.


---
## Step 12: Run Seeds 123 and 456

Seed 42 already done (F1=0.8959). Each run ~7 hrs.


In [7]:
all_results = []

# Seed 42 — already completed
all_results.append({
    'seed': 42, 'macro_f1': 0.8959, 'mcc': 0.7949, 'auc_roc': 0.9695,
    'accuracy': 0.0, 'train_runtime_s': 0, 'note': 'from original run'
})
print("Seed 42: F1=0.8959 | MCC=0.7949 | AUC=0.9695 (from original run)")

for seed in TRAIN_SEEDS:
    result = run_single_seed_cb2(train_df, val_df, test_df, train_seed=seed)
    all_results.append(result)


Seed 42: F1=0.8959 | MCC=0.7949 | AUC=0.9695 (from original run)

  MULTI-SEED CB2 — train_seed=123
  Split seed=42 (fixed) | Gemma-2-9B | r=16


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

[2] Injecting GeneralisedLoRALayer...
  ✓ Replaced 168 layers
[3] Training (seed=456)...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss,Accuracy,F1,Mcc
100,0.435600,0.561350,0.843018,0.813455,0.647148
200,0.426000,0.435192,0.854004,0.826699,0.673484
300,0.359100,0.301477,0.871206,0.863723,0.736615
400,0.348100,0.284306,0.873374,0.865134,0.736407
500,0.299400,0.341371,0.880023,0.865084,0.732041
600,0.364100,0.292849,0.866291,0.859685,0.733489
700,0.330600,0.275986,0.889563,0.879775,0.760212
800,0.323200,0.268918,0.890864,0.881434,0.763799
900,0.317200,0.280681,0.882191,0.875401,0.760332
1000,0.293100,0.270898,0.876843,0.869938,0.750160


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

    Time: 21754s (6.04 hrs)
[4] Evaluating...

    Seed 456: F1=0.8952 | MCC=0.7941 | AUC=0.9691


---
## Step 13: Multi-Seed Summary — CB2

In [8]:
print('\n' + '═'*70)
print('  MULTI-SEED VALIDATION — Gemma-2-9B CB2 Stage 4')
print('  Formula: Wf = W(I + UV) + AB | r=16')
print('  Split seed: 42 (fixed) | Train seeds: 42, 123, 456')
print('═'*70)

f1_values  = [r['macro_f1'] for r in all_results]
mcc_values = [r['mcc'] for r in all_results]
auc_values = [r['auc_roc'] for r in all_results]

print(f'\n  {"Seed":<8} {"Macro F1":<12} {"MCC":<12} {"AUC-ROC":<12}')
print(f'  {"─"*44}')
for r in all_results:
    note = ' (original)' if r['seed'] == 42 else ''
    print(f'  {r["seed"]:<8} {r["macro_f1"]:<12.4f} {r["mcc"]:<12.4f} {r["auc_roc"]:<12.4f}{note}')

mean_f1  = np.mean(f1_values);  std_f1  = np.std(f1_values)
mean_mcc = np.mean(mcc_values); std_mcc = np.std(mcc_values)
mean_auc = np.mean(auc_values); std_auc = np.std(auc_values)

print(f'\n  {"─"*44}')
print(f'  {"Mean":<8} {mean_f1:<12.4f} {mean_mcc:<12.4f} {mean_auc:<12.4f}')
print(f'  {"Std":<8} {std_f1:<12.4f} {std_mcc:<12.4f} {std_auc:<12.4f}')

print(f'\n  ── Comparison ──')
print(f'  Stage 2 (seed 42): F1=0.8885 | AUC=0.9654')
print(f'  Stage 3 (seed 42): F1=0.8948 | AUC=0.9686')
print(f'  Stage 4 mean±std : F1={mean_f1:.4f}±{std_f1:.4f} | AUC={mean_auc:.4f}±{std_auc:.4f}')
print(f'  Stage 4 min      : F1={min(f1_values):.4f}')

if min(f1_values) > 0.8885:
    print(f'\n  ★ ALL seeds beat Stage 2 (0.8885)')
if min(f1_values) > 0.8948:
    print(f'  ★ ALL seeds beat Stage 3 (0.8948)')
elif mean_f1 > 0.8948:
    print(f'  ✓ Mean beats Stage 3 (0.8948)')

# Save
summary_df = pd.DataFrame(all_results)
summary_df.loc[len(summary_df)] = {
    'seed': 'mean', 'macro_f1': mean_f1, 'mcc': mean_mcc,
    'auc_roc': mean_auc, 'accuracy': 0, 'train_runtime_s': 0}
summary_df.loc[len(summary_df)] = {
    'seed': 'std', 'macro_f1': std_f1, 'mcc': std_mcc,
    'auc_roc': std_auc, 'accuracy': 0, 'train_runtime_s': 0}
summary_path = os.path.join(OUTPUT_DIR, 'multi_seed_summary_CB2_FINAL.csv')
summary_df.to_csv(summary_path, index=False)
print(f'\n✓ Saved → {summary_path}')



══════════════════════════════════════════════════════════════════════
  MULTI-SEED VALIDATION — Gemma-2-9B CB2 Stage 4
  Formula: Wf = W(I + UV) + AB | r=16
  Split seed: 42 (fixed) | Train seeds: 42, 123, 456
══════════════════════════════════════════════════════════════════════

  Seed     Macro F1     MCC          AUC-ROC     
  ────────────────────────────────────────────
  42       0.8959       0.7949       0.9695       (original)
  123      0.8928       0.7887       0.9688      
  456      0.8952       0.7941       0.9691      

  ────────────────────────────────────────────
  Mean     0.8946       0.7926       0.9691      
  Std      0.0013       0.0027       0.0003      

  ── Comparison ──
  Stage 2 (seed 42): F1=0.8885 | AUC=0.9654
  Stage 3 (seed 42): F1=0.8948 | AUC=0.9686
  Stage 4 mean±std : F1=0.8946±0.0013 | AUC=0.9691±0.0003
  Stage 4 min      : F1=0.8928

  ★ ALL seeds beat Stage 2 (0.8885)

✓ Saved → /content/drive/MyDrive/Results/CB2_GeneralisedLoRA_Gemma2_9B_Mult